# Stacking CoxMLP neural network meta-learner (CpG-Only) - `[PROJECT_ROOT]`

This notebook trains and evaluates the Stage-2 stacking neural network ensemble model (`CoxMLP`) using PyTorch on CpG-only data. It:
1. Loads the Stage-1 CpG-only out-of-fold and refitted base model predictions.
2. Defines the `CoxMLP` neural network stacker mapping base model outputs to a single continuous risk score.
3. Uses the shape-fixed Cox proportional hazards negative log partial likelihood loss (DeepSurv loss).
4. Retrains the final ensemble model on combined train + val stacking datasets.
5. Exports final predictions to predictions/ with alignment-safe index matching.


In [ ]:
# ==========================================================
# 1. IMPORTS & CONFIGURATION


In [ ]:
import sys
from pathlib import Path

# Resolve repository root and import config
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent:
    if (_repo_root / "config.py").exists():
        if str(_repo_root) not in sys.path:
            sys.path.insert(0, str(_repo_root))
        break
    _repo_root = _repo_root.parent

import config

# ==========================================================
import os
import sys
import gc
import json
import warnings
import logging
import datetime as dt
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple, Optional, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.inspection import permutation_importance

# Scikit-Survival
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.svm import FastKernelSurvivalSVM
from sksurv.metrics import concordance_index_censored

# Lifelines
from lifelines import WeibullAFTFitter
from lifelines.utils import concordance_index

# tqdm
from tqdm.auto import tqdm

# Pycox
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchtuples as tt
from torchtuples import optim as ttoptim
from torchtuples.callbacks import EarlyStopping
from pycox.models import MTLR as PycoxMTLR
from torchtuples.practical import MLPVanilla

warnings.filterwarnings("ignore")

# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    "model": {
        "hidden_layers": [256, 128, 64],
        "dropout": 0.4,
        "lr": 4.396e-5,
        "weight_decay": 6.943e-5
    },
    "training": {
        "max_epochs": 2 if os.environ.get("EPICLOCK_SMOKE") == "1" else 150,
        "patience": 15,
        "seed": 42
    }
}

# Set seeds for reproducibility
np.random.seed(CONFIG["training"]["seed"])
torch.manual_seed(CONFIG["training"]["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed(CONFIG["training"]["seed"])

RUN_ID = "V3_OPTUNA_SS_CPG"

# Directories
BASE_DIR = config.PROJECT_ROOT
DATA_DIR = config.DATA_DIR       # shared raw data
MODEL_DIR = BASE_DIR / "saved_models"
PRED_DIR = BASE_DIR / "predictions"
SCALER_DIR = BASE_DIR / "scalers"
LOG_DIR = BASE_DIR / "logs"

# Ensure all folders exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
SCALER_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Set up logging to both stdout and a file
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_DIR / "pipeline_cpg.log", mode='a', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger()

# Fast run parameter
FAST_RUN = True

N_SPLITS = 5

logger.info(f"Pipeline Initialized in {config.PROJECT_ROOT}")
logger.info(f"FAST_RUN mode: {FAST_RUN}")
logger.info(f"RUN_ID: {RUN_ID}")


In [ ]:
# ==========================================================
# 11. Stage-2 CoxMLP Stacking Data Prep


In [ ]:
# ==========================================================
logger.info("Starting Stage-2 CoxMLP Stacking...")

# Read predictions
train_df_stack = pd.read_csv(PRED_DIR / f"stacking_train_{RUN_ID}.csv")
val_df_stack   = pd.read_csv(PRED_DIR / f"stacking_val_{RUN_ID}.csv")
test_df_stack  = pd.read_csv(PRED_DIR / f"stacking_test_{RUN_ID}.csv")
whi_df_stack   = pd.read_csv(PRED_DIR / f"stacking_whi_{RUN_ID}.csv")

TIME_COL     = "time"
EVENT_COL    = "event"
# Exclude non-input columns
EXCLUDE_COLS = [TIME_COL, EVENT_COL, "patient_id", "age", "sex"]

FEATURE_COLS = [c for c in train_df_stack.columns if c not in EXCLUDE_COLS]
logger.info(f"Stacking input features: {FEATURE_COLS}")


In [ ]:
# ==========================================================
# 11b. QUANTILE NORMALIZATION OF STACKED RISK SCORES


In [ ]:
# ==========================================================
# Apply QuantileTransformer (output_distribution='normal') to the 5 base-model
# risk score columns. Fit on training OOF risk scores only, then transform
# val/test/whi using the same fitted transformer.
#
# This ensures each model's risk distribution is mapped to a standard normal,
# making the ensemble inputs comparable in scale and distribution.


In [ ]:
# ==========================================================
logger.info("Applying StandardScaler to stacked risk scores...")

scaler = StandardScaler()

# Fit on training data only (leakage-free)
train_df_stack[FEATURE_COLS] = scaler.fit_transform(
    train_df_stack[FEATURE_COLS].values
)
logger.info(f"  Fitted StandardScaler on training OOF risk scores ({len(train_df_stack)} samples)")

# Transform validation, test, and external WHI splits
val_df_stack[FEATURE_COLS] = scaler.transform(val_df_stack[FEATURE_COLS].values)
test_df_stack[FEATURE_COLS] = scaler.transform(test_df_stack[FEATURE_COLS].values)
whi_df_stack[FEATURE_COLS] = scaler.transform(whi_df_stack[FEATURE_COLS].values)
logger.info("  Transformed val/test/whi risk scores using fitted scaler")

# Save the fitted StandardScaler for reproducibility
scaler_save_path = SCALER_DIR / f"scaler_stacking_{RUN_ID}.joblib"
dump(scaler, scaler_save_path)
logger.info(f"  Saved StandardScaler -> {scaler_save_path}")


def sort_by_time_descending(df):
    return df.sort_values(by=TIME_COL, ascending=False).reset_index(drop=True)

class SurvivalDataset(Dataset):
    def __init__(self, df, feature_cols, time_col, event_col):
        self.X     = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.time  = torch.tensor(df[time_col].values,     dtype=torch.float32)
        self.event = torch.tensor(df[event_col].values,    dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.time[idx], self.event[idx]

n_features = len(FEATURE_COLS)
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
logger.info(f"Using device for PyTorch: {device}")


In [ ]:
# ==========================================================
# 12. PyTorch CoxMLP Model Architecture


In [ ]:
# ==========================================================
class CoxMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers=[128, 64, 128], dropout=0.1):
        super().__init__()
        layers_list = []
        in_dim = input_dim
        for h_dim in hidden_layers:
            layers_list.append(nn.Linear(in_dim, h_dim))
            layers_list.append(nn.ReLU())
            if dropout > 0:
                layers_list.append(nn.Dropout(dropout))
            in_dim = h_dim
        output_layer = nn.Linear(in_dim, 1, bias=False)
        nn.init.zeros_(output_layer.weight)
        layers_list.append(output_layer)
        self.network = nn.Sequential(*layers_list)
        
    def forward(self, x):
        return self.network(x)


In [ ]:
# ==========================================================
# 13. Loss Function & Evaluation Metric Helpers


In [ ]:
# ==========================================================
def cox_ph_loss_sorted(risk_scores, events):
    risk_scores = risk_scores.view(-1)
    events = events.view(-1)
    exp_risk = torch.exp(risk_scores)
    risk_set_sum = torch.cumsum(exp_risk, dim=0)
    log_risk_set_sum = torch.log(risk_set_sum + 1e-7)
    uncensored_loss = events * (log_risk_set_sum - risk_scores)
    num_events = torch.sum(events) + 1e-7
    return torch.sum(uncensored_loss) / num_events

def cox_ph_loss(risk_scores, times, events):
    times_sorted, indices = torch.sort(times, descending=True)
    risk_scores_sorted = risk_scores[indices]
    events_sorted = events[indices]
    return cox_ph_loss_sorted(risk_scores_sorted, events_sorted)

def compute_cindex(model, loader, device):
    model.eval()
    all_risk, all_time, all_event = [], [], []
    with torch.no_grad():
        for X, t, e in loader:
            X = X.to(device, non_blocking=True)
            risk_scores = model(X).detach().cpu().numpy().ravel()
            all_risk.extend(risk_scores)
            all_time.extend(t.numpy())
            all_event.extend(e.numpy())
    all_risk = np.array(all_risk)
    all_time = np.array(all_time)
    all_event = np.array(all_event)
    valid = ~np.isnan(all_risk)
    if valid.sum() < 10:
        return 0.5
    return concordance_index(all_time[valid], -all_risk[valid], all_event[valid])


In [ ]:
# ==========================================================
# 14. CoxMLP Training Function


In [ ]:
# ==========================================================
def train_coxmlp_model(params, train_loader, val_loader, device, n_features, max_epochs=100, patience=10, whi_loader=None):
    model = CoxMLP(
        input_dim = n_features,
        hidden_layers = params["hidden_layers"],
        dropout = params["dropout"]
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

    best_val_c = -np.inf
    best_state = None
    no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0.0
        num_batches = 0
        for X, t, e in train_loader:
            X = X.to(device, non_blocking=True)
            t = t.to(device, non_blocking=True)
            e = e.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            risk_scores = model(X)
            if torch.isnan(risk_scores).any():
                continue
            loss = cox_ph_loss(risk_scores, t, e)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1

        scheduler.step()

        avg_loss = epoch_loss / num_batches if num_batches > 0 else np.nan
        train_c = compute_cindex(model, train_loader, device)
        val_c = compute_cindex(model, val_loader, device)
        
        if whi_loader is not None:
            whi_c = compute_cindex(model, whi_loader, device)
            logger.info(f"Epoch {epoch+1:03d}/{max_epochs:03d} | Train Loss: {avg_loss:.4f} | Train C-Index: {train_c:.4f} | Val C-Index: {val_c:.4f} | WHI C-Index: {whi_c:.4f}")
        else:
            logger.info(f"Epoch {epoch+1:03d}/{max_epochs:03d} | Train Loss: {avg_loss:.4f} | Train C-Index: {train_c:.4f} | Val C-Index: {val_c:.4f}")

        if val_c > best_val_c:
            best_val_c = val_c
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                logger.info(f"Early stopping triggered at Epoch {epoch+1} (patience={patience})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_c


In [ ]:
# ==========================================================
# 15. BASELINE CoxMLP TRAINING (Train on Train, validate on Val)


In [ ]:
# ==========================================================
train_df_sorted = sort_by_time_descending(train_df_stack)
val_df_sorted   = sort_by_time_descending(val_df_stack)

train_dataset = SurvivalDataset(train_df_sorted, FEATURE_COLS, TIME_COL, EVENT_COL)
val_dataset   = SurvivalDataset(val_df_sorted,   FEATURE_COLS, TIME_COL, EVENT_COL)
whi_dataset   = SurvivalDataset(whi_df_stack,    FEATURE_COLS, TIME_COL, EVENT_COL)

train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)
val_loader   = DataLoader(val_dataset,   batch_size=len(val_dataset),   shuffle=False)
whi_loader   = DataLoader(whi_dataset,   batch_size=len(whi_dataset),   shuffle=False)

logger.info(f"Training baseline CoxMLP model with parameters: {CONFIG['model']}")
logger.info(f"NOTE: Ensemble inputs have been scaled (StandardScaler)")
baseline_params = CONFIG["model"]
baseline_model, val_c = train_coxmlp_model(
    baseline_params, train_loader, val_loader, device, n_features,
    max_epochs=CONFIG["training"]["max_epochs"],
    patience=CONFIG["training"]["patience"],
    whi_loader=whi_loader
)
logger.info(f"Baseline CoxMLP Val C-index: {val_c:.4f}")


In [ ]:
# ==========================================================
# 16. RETRAIN CoxMLP ON COMBINED TRAIN + VAL


In [ ]:
# ==========================================================
combined_df_stack = pd.concat([train_df_stack, val_df_stack], ignore_index=True)
combined_df_sorted = sort_by_time_descending(combined_df_stack)

combined_ds = SurvivalDataset(combined_df_sorted, FEATURE_COLS, TIME_COL, EVENT_COL)
combined_loader = DataLoader(combined_ds, batch_size=len(combined_ds), shuffle=False)
val_loader_final = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

logger.info(f"Retraining final CoxMLP model with parameters: {CONFIG['model']}")
final_coxmlp, _ = train_coxmlp_model(
    CONFIG["model"], combined_loader, val_loader_final,
    device, n_features,
    max_epochs=CONFIG["training"]["max_epochs"],
    patience=CONFIG["training"]["patience"],
    whi_loader=whi_loader
)


In [ ]:
# ==========================================================
# 17. FINAL EVALUATION & EXPORT ENSEMBLE PREDICTIONS


In [ ]:
# ==========================================================
test_dataset = SurvivalDataset(test_df_stack, FEATURE_COLS, TIME_COL, EVENT_COL)
whi_dataset  = SurvivalDataset(whi_df_stack,  FEATURE_COLS, TIME_COL, EVENT_COL)

loaders = {
    "Train": DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False),
    "Val"  : DataLoader(val_dataset,   batch_size=len(val_dataset),   shuffle=False),
    "Test" : DataLoader(test_dataset,  batch_size=len(test_dataset),  shuffle=False),
    "WHI"  : DataLoader(whi_dataset,   batch_size=len(whi_dataset),   shuffle=False),
}

logger.info("\n--- FINAL ENSEMBLE C-INDEX RESULTS ---")
ensemble_cis = {}
for split, loader in loaders.items():
    c = compute_cindex(final_coxmlp, loader, device)
    ensemble_cis[split] = c
    logger.info(f"  Ensemble {split:6s}: {c:.4f}")

# Save the ensemble model state dict
save_path = MODEL_DIR / "coxmlp_surv_best.pt"
torch.save({
    "model_state":  final_coxmlp.state_dict(),
    "feature_cols": FEATURE_COLS,
    "best_params":  CONFIG["model"],
}, save_path)
logger.info(f"Saved checkpoint to {save_path}")

# Export predictions containing ensemble output (fixed order alignment!)
def save_ensemble_predictions(df, split_name):
    final_coxmlp.eval()
    ds = SurvivalDataset(df, FEATURE_COLS, TIME_COL, EVENT_COL)
    loader = DataLoader(ds, batch_size=len(ds), shuffle=False)
    with torch.no_grad():
        for X, _, _ in loader:
            risk_scores = final_coxmlp(X.to(device)).detach().cpu().numpy().ravel()
            break
        
    out_df = df.copy()
    out_df["ensemble"] = risk_scores
    out_csv_path = PRED_DIR / f"stacking_predictions_{split_name}_{RUN_ID}.csv"
    out_df.to_csv(out_csv_path, index=False)
    logger.info(f"Ensemble predictions saved -> {out_csv_path}")

save_ensemble_predictions(train_df_stack, "train")
save_ensemble_predictions(val_df_stack, "val")
save_ensemble_predictions(test_df_stack, "test")
save_ensemble_predictions(whi_df_stack, "whi")

logger.info("Pipeline Complete!")
